# Business Analysis

In [0]:
from utils.logger import get_logger
from etl_config.constants_config import GOLD

logger = get_logger("gold_marts")

In [0]:
spark.sql(f"""
    create or replace table {GOLD}.agg_sales_by_month
    comment 'Gold KPI: monthly sales aggregated by customer, division and category'
    as
    select
        dd_month.date_id,
        dc.customer_sk,
        dc.division_id,
        dp.category_id,
        cast(sum(fol.revenue) as decimal(14,2)) as total_revenue,
        cast(sum(fol.cost) as decimal(14,2)) as total_cost,
        cast(sum(fol.margin) as decimal(14,2)) as total_margin
    from {GOLD}.fact_order_lines fol
    join {GOLD}.fact_orders fo on fol.order_id = fo.order_id
    join {GOLD}.dim_date dd on fo.date_id = dd.date_id
    join {GOLD}.dim_date dd_month on dd_month.year = dd.year and dd_month.month = dd.month and dd_month.is_first_day_of_month = true
    join {GOLD}.dim_customers dc on fo.customer_sk = dc.customer_sk and dc.is_current = true
    join {GOLD}.dim_products dp on fol.product_sk = dp.product_sk and dp.is_current = true
    group by dd_month.date_id, dc.customer_sk, dc.division_id, dp.category_id
""")
 
row_count = spark.table(f"{GOLD}.agg_sales_by_month").count()
logger.info(f"agg_sales_by_month rebuilt: {row_count} rows")

In [0]:
full_table_name = f"{GOLD}.agg_sales_by_month"
 
# Set table properties
spark.sql(f"""
    ALTER TABLE {full_table_name}
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true'
    )
""")
logger.info(f"Set table properties for {full_table_name}")
 
agg_sales_by_month_comments = {
    "date_id": "FK to dim_date (first day of the month)",
    "customer_sk": "FK to dim_customers",
    "division_id": "FK to dim_division",
    "category_id": "FK to dim_category",
    "total_revenue": "Sum of order line revenue for the customer/division/category/month",
    "total_cost": "Sum of order line cost for the customer/division/category/month",
    "total_margin": "Sum of order line margin (revenue - cost)",
}
 
for column, comment in agg_sales_by_month_comments.items():
    spark.sql(
        f"COMMENT ON COLUMN {full_table_name}.{column} IS :comment",
        args={"comment": comment}
    )
 
logger.info(f"Commented columns for {full_table_name}")

In [0]:
spark.sql(f"""
    create or replace table {GOLD}.agg_customer_growth_by_month
    comment 'Gold KPI: monthly customer growth with new, active and YTD counts'
    as
    with monthly_customers as (
        select distinct date_id, customer_sk
        from {GOLD}.agg_sales_by_month
    ),
    customer_first_order as (
        select customer_sk, min(date_id) as first_order_date_id
        from monthly_customers
        group by customer_sk
    ),
    monthly_agg as (
        select
            dd.year, dd.month, mc.date_id,
            count(distinct mc.customer_sk) as active_customers,
            count(distinct case
                when cfo.first_order_date_id = mc.date_id
                then mc.customer_sk
            end) as new_customers
        from monthly_customers mc
        join {GOLD}.dim_date dd on mc.date_id = dd.date_id
        join customer_first_order cfo on mc.customer_sk = cfo.customer_sk
        group by dd.year, dd.month, mc.date_id
    )
    select ma.date_id,
           ma.new_customers, ma.active_customers,
           sum(ma.new_customers) over (
                partition by ma.year order by ma.month
                rows between unbounded preceding and current row
            ) as ytd_customers
    from monthly_agg ma
""")
 
row_count = spark.table(f"{GOLD}.agg_customer_growth_by_month").count()
logger.info(f"agg_customer_growth_by_month rebuilt: {row_count} rows")

In [0]:
full_table_name = f"{GOLD}.agg_customer_growth_by_month"
 
# Set table properties
spark.sql(f"""
    ALTER TABLE {full_table_name}
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true'
    )
""")
logger.info(f"Set table properties for {full_table_name}")
 
agg_customer_growth_by_month_comments = {
    "date_id": "FK to dim_date (first day of the month)",
    "new_customers": "Customers whose first-ever order line (agg_sales_by_month) fell in this month",
    "active_customers": "Distinct customers with at least one order line in this month",
    "ytd_customers": "Cumulative sum of new_customers within the calendar year, up to this month",
}
 
for column, comment in agg_customer_growth_by_month_comments.items():
    spark.sql(
        f"COMMENT ON COLUMN {full_table_name}.{column} IS :comment",
        args={"comment": comment}
    )
 
logger.info(f"Commented columns for {full_table_name}")

### 1. Analyze sales figures and compare them year-over-year

In [0]:
spark.sql(f"""
    create or replace view {GOLD}.vw_sales_yoy
    comment 'Monthly sales revenue YoY'
    as
    with monthly_revenue as (
        select a.date_id, year, month, month_name, sum(total_revenue) as revenue
        from {GOLD}.agg_sales_by_month a
        join {GOLD}.dim_date dd on a.date_id = dd.date_id
        group by a.date_id, year, month, month_name
    )
    select
        curr.date_id,
        curr.revenue as current_year_revenue,
        prev.revenue as last_year_revenue,
        curr.revenue - prev.revenue as yoy_change,
        round((curr.revenue - prev.revenue) / nullif(prev.revenue, 0) * 100, 2) as yoy_pct_change
    from monthly_revenue curr
    left join monthly_revenue prev
        on curr.month = prev.month
        and curr.year = prev.year + 1
""")
 
logger.info(f"Created view {GOLD}.vw_sales_yoy")

### 2. Measure and track sales margins

In [0]:
spark.sql(f"""
    create or replace view {GOLD}.vw_sales_margin_tracking
    comment 'Monthly margin tracking'
    as
    with order_level_freight as (
        select
            year(order_date) as year,
            month(order_date) as month,
            sum(freight) as total_freight
        from {GOLD}.fact_orders
        group by year(order_date), month(order_date)
    ),
    monthly_gross_margin as (
        select
            a.date_id, dd.year, dd.quarter, dd.month, dd.month_name,
            sum(a.total_revenue) as total_revenue,
            sum(a.total_cost) as total_cost,
            sum(a.total_margin) as gross_margin
        from {GOLD}.agg_sales_by_month a
        join {GOLD}.dim_date dd on a.date_id = dd.date_id
        group by a.date_id, dd.year, dd.quarter, dd.month, dd.month_name
    )
    select
        m.date_id,
        m.total_revenue,
        m.total_cost,
        m.gross_margin,
        round(m.gross_margin / nullif(m.total_revenue, 0) * 100, 2) as gross_margin_pct,
        f.total_freight,
        m.gross_margin - f.total_freight as net_margin,
        round((m.gross_margin - f.total_freight) / nullif(m.total_revenue, 0) * 100, 2) as net_margin_pct
    from monthly_gross_margin m
    left join order_level_freight f
        on m.year = f.year and m.month = f.month
""")
 
logger.info(f"Created view {GOLD}.vw_sales_margin_tracking")

### 3. Analyze sales data by various dimensions (Region, Country, Division, Product Line, Product Group, Customer)

In [0]:
spark.sql(f"""
    create or replace view {GOLD}.vw_sales_by_region_product_line
    comment 'Monthly revenue and margin by division and product line (division_id FK to dim_division, category_id FK to dim_category)'
    as
    select
        a.date_id,
        a.division_id,
        a.category_id,
        sum(a.total_revenue) as total_revenue,
        sum(a.total_margin) as total_margin
    from {GOLD}.agg_sales_by_month a
    group by a.date_id, a.division_id, a.category_id
""")
 
logger.info(f"Created view {GOLD}.vw_sales_by_region_product_line")

### By Customer

In [0]:
spark.sql(f"""
    create or replace view {GOLD}.vw_sales_by_customer
    comment 'Monthly revenue, margin and order count by customer (group by country/company_name in Power BI via dim_customers relation)'
    as
    with monthly_customer_sales as (
        select
            date_id,
            customer_sk,
            sum(total_revenue) as total_revenue,
            sum(total_margin) as total_margin
        from {GOLD}.agg_sales_by_month
        group by date_id, customer_sk
    ),
    monthly_customer_orders as (
        select
            dd_month.date_id,
            fo.customer_sk,
            count(distinct fo.order_id) as order_count
        from {GOLD}.fact_orders fo
        join {GOLD}.dim_date dd
            on fo.date_id = dd.date_id
        join {GOLD}.dim_date dd_month
            on dd_month.year = dd.year and dd_month.month = dd.month and dd_month.is_first_day_of_month = true
        group by dd_month.date_id, fo.customer_sk
    )
    select
        s.date_id,
        s.customer_sk,
        s.total_revenue,
        s.total_margin,
        o.order_count
    from monthly_customer_sales s
    left join monthly_customer_orders o
        on s.date_id = o.date_id and s.customer_sk = o.customer_sk
""")
logger.info(f"Created view {GOLD}.vw_sales_by_customer")

### By Product

In [0]:
spark.sql(f"""
    create or replace view {GOLD}.vw_sales_by_product
    comment 'Monthly revenue and margin by product (product_sk FK to dim_products)'
    as
    select
        dd_month.date_id,
        fol.product_sk,
        cast(sum(fol.revenue) as decimal(14,2)) as total_revenue,
        cast(sum(fol.margin) as decimal(14,2)) as total_margin,
        round(sum(fol.margin) / nullif(sum(fol.revenue), 0) * 100, 2) as margin_pct
    from {GOLD}.fact_order_lines fol
    join {GOLD}.fact_orders fo on fol.order_id = fo.order_id
    join {GOLD}.dim_date dd on fo.date_id = dd.date_id
    join {GOLD}.dim_date dd_month
        on dd_month.year = dd.year and dd_month.month = dd.month and dd_month.is_first_day_of_month = true
    group by dd_month.date_id, fol.product_sk
""")

logger.info(f"Created view {GOLD}.vw_sales_by_product")

### 4. Compute average sales per transaction and per customer

In [0]:
spark.sql(f"""
    create or replace view {GOLD}.vw_avg_revenue_per_transaction_customer
    comment 'Monthly average revenue per transaction and per distinct customer'
    as
    with monthly_sales as (
        select
            date_id,
            sum(total_revenue) as total_revenue,
            count(distinct customer_sk) as total_customers
        from {GOLD}.agg_sales_by_month
        group by date_id
    ),
    monthly_orders as (
        select
            dd_month.date_id,
            count(distinct fo.order_id) as total_orders
        from {GOLD}.fact_orders fo
        join {GOLD}.dim_date dd on fo.date_id = dd.date_id
        join {GOLD}.dim_date dd_month
            on dd_month.year = dd.year
                and dd_month.month = dd.month
                and dd_month.is_first_day_of_month = true
        group by dd_month.date_id
    )
    select
        s.date_id,
        s.total_revenue,
        o.total_orders,
        s.total_customers,
        round(s.total_revenue / nullif(o.total_orders, 0), 2) as avg_revenue_per_transaction,
        round(s.total_revenue / nullif(s.total_customers, 0), 2) as avg_revenue_per_customer
    from monthly_sales s
    left join monthly_orders o
        on s.date_id = o.date_id
""")
 
logger.info(f"Created view {GOLD}.vw_avg_revenue_per_transaction_customer")

### 5. Customer growth — YTD vs LYTD

In [0]:
spark.sql(f"""
    create or replace view {GOLD}.vw_customer_growth_ytd_vs_lytd
    comment 'Customer growth YTD compared to the LYTD'
    as
    with growth_with_date as (
        select
            g.date_id, dd.year, dd.month,
            g.new_customers, g.active_customers, g.ytd_customers
        from {GOLD}.agg_customer_growth_by_month g
        join {GOLD}.dim_date dd on g.date_id = dd.date_id
    )
    select
        curr.date_id,
        curr.new_customers,
        curr.active_customers,
        curr.ytd_customers,
        prev.ytd_customers as lytd_customers,
        curr.ytd_customers - prev.ytd_customers as ytd_vs_lytd_change,
        round((curr.ytd_customers - prev.ytd_customers) / nullif(prev.ytd_customers, 0) * 100, 2) as ytd_vs_lytd_pct
    from growth_with_date curr
    left join growth_with_date prev
        on curr.month = prev.month and curr.year = prev.year + 1
""")
 
logger.info(f"Created view {GOLD}.vw_customer_growth_ytd_vs_lytd")